<a href="https://colab.research.google.com/github/comsadavid/AI-solutions/blob/main/competitions/mlcompete/Analiza_%C8%99i_clasificarea_riscurilor_%C8%99i_simptomelor_Parkinson.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import roc_auc_score
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, GridSearchCV

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

from catboost import CatBoostClassifier


In [ ]:
df_train = pd.read_csv('train.csv')
df_test = pd.read_csv('test.csv')

In [ ]:
df_train.head()

In [ ]:
df_train.dtypes

In [ ]:
def encode_columns(df_train, df_test, columns):
    ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore', dtype=int)
    encoded_train_list = ohe.fit_transform(df_train[columns])
    encoded_test_list = ohe.transform(df_test[columns])

    encoded_df_train = pd.DataFrame(encoded_train_list, columns=ohe.get_feature_names_out())
    encoded_df_test = pd.DataFrame(encoded_test_list, columns=ohe.get_feature_names_out())

    df_train[encoded_df_train.columns] = encoded_df_train
    df_test[encoded_df_test.columns] = encoded_df_test

    df_train.drop(columns, axis=1, inplace=True)
    df_test.drop(columns, axis=1, inplace=True)

def process(df_train, df_test):
    encode_list = ['EyeColor']
    encode_columns(df_train, df_test, encode_list)

    drop_list = ['DoctorInCharge']
    df_train.drop(drop_list, axis=1, inplace=True)
    df_test.drop(drop_list, axis=1, inplace=True)

In [ ]:
process(df_train, df_test)
df_train.head()

In [ ]:
X = df_train.drop(['PatientID', 'Diagnosis'], axis=1)
y = df_train['Diagnosis']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Convert scaled arrays back to DataFrame for consistency with other models
#X_train = pd.DataFrame(X_train, columns=X_train_raw.columns)
#X_test = pd.DataFrame(X_test, columns=X_test_raw.columns)

type(X_train)

In [ ]:
def evaluate_model(model):
    # For AUC, it's better to use predict_proba for probability scores
    y_train_proba = model.predict_proba(X_train)[:, 1]
    y_test_proba = model.predict_proba(X_test)[:, 1]

    print(f"Train ROC AUC: {roc_auc_score(y_train, y_train_proba)}, Test ROC AUC: {roc_auc_score(y_test, y_test_proba)}")

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
evaluate_model(rf)

In [ ]:
ctb = CatBoostClassifier(iterations=600, max_depth=8, learning_rate=0.03)
ctb.fit(X_train, y_train)
evaluate_model(ctb)

0:	learn: 0.6662064	total: 35.9ms	remaining: 21.5s
1:	learn: 0.6405095	total: 60.6ms	remaining: 18.1s
2:	learn: 0.6080370	total: 84.8ms	remaining: 16.9s
3:	learn: 0.5867454	total: 111ms	remaining: 16.5s
4:	learn: 0.5622911	total: 138ms	remaining: 16.4s
5:	learn: 0.5425572	total: 168ms	remaining: 16.7s
6:	learn: 0.5184255	total: 193ms	remaining: 16.4s
7:	learn: 0.5029537	total: 219ms	remaining: 16.2s
8:	learn: 0.4804941	total: 248ms	remaining: 16.3s
9:	learn: 0.4646097	total: 274ms	remaining: 16.2s
10:	learn: 0.4509402	total: 299ms	remaining: 16s
11:	learn: 0.4353910	total: 325ms	remaining: 15.9s
12:	learn: 0.4223762	total: 350ms	remaining: 15.8s
13:	learn: 0.4064851	total: 371ms	remaining: 15.5s
14:	learn: 0.4003792	total: 407ms	remaining: 15.9s
15:	learn: 0.3867124	total: 433ms	remaining: 15.8s
16:	learn: 0.3784970	total: 461ms	remaining: 15.8s
17:	learn: 0.3650053	total: 495ms	remaining: 16s
18:	learn: 0.3546032	total: 526ms	remaining: 16.1s
19:	learn: 0.3463795	total: 547ms	remainin

In [ ]:
ctb_param_grid = {
    "iterations": [300, 600],
    "learning_rate": [0.03, 0.1],
    "depth": [6, 8],
    "l2_leaf_reg": [3, 7],
}

ctb_grid_search = GridSearchCV(
    CatBoostClassifier(),
    ctb_param_grid,
    scoring="roc_auc",
    n_jobs=-1
)

ctb_grid_search.fit(X_train, y_train)

ctb_best_params = ctb_grid_search.best_params_

print(ctb_best_params)

0:	learn: 0.6692325	total: 63.6ms	remaining: 38.1s
1:	learn: 0.6406092	total: 77.5ms	remaining: 23.2s
2:	learn: 0.6136122	total: 90.5ms	remaining: 18s
3:	learn: 0.5969352	total: 104ms	remaining: 15.5s
4:	learn: 0.5774172	total: 117ms	remaining: 13.9s
5:	learn: 0.5610968	total: 130ms	remaining: 12.9s
6:	learn: 0.5403296	total: 143ms	remaining: 12.1s
7:	learn: 0.5271488	total: 156ms	remaining: 11.6s
8:	learn: 0.5074679	total: 169ms	remaining: 11.1s
9:	learn: 0.4923147	total: 183ms	remaining: 10.8s
10:	learn: 0.4774427	total: 196ms	remaining: 10.5s
11:	learn: 0.4630778	total: 215ms	remaining: 10.5s
12:	learn: 0.4536856	total: 239ms	remaining: 10.8s
13:	learn: 0.4424687	total: 252ms	remaining: 10.6s
14:	learn: 0.4369670	total: 268ms	remaining: 10.5s
15:	learn: 0.4241770	total: 281ms	remaining: 10.3s
16:	learn: 0.4167458	total: 294ms	remaining: 10.1s
17:	learn: 0.4098533	total: 307ms	remaining: 9.94s
18:	learn: 0.4015051	total: 320ms	remaining: 9.8s
19:	learn: 0.3935563	total: 333ms	remaini

In [ ]:
def solve(model):
    # Drop 'PatientID' from df_test as it's not a feature
    # Ensure X_pred has the same columns as X (from df_train) before scaling
    X_pred = df_test[X.columns] # Use the columns from the original X DataFrame
    X_pred_scaled = scaler.transform(X_pred)
    y_pred = model.predict(X_pred_scaled)

    output_3 = pd.DataFrame({'PatientID': df_test['PatientID'], 'subtaskID': 'Task3', 'Answer': y_pred})

    # Initialize output_1 and output_2 as empty DataFrames if not defined
    # This prevents NameError if they are meant to be generated by other tasks later
    global output_1, output_2
    if 'output_1' not in globals():
        output_1 = pd.DataFrame(columns=['PatientID', 'subtaskID', 'Answer'])
    if 'output_2' not in globals():
        output_2 = pd.DataFrame(columns=['PatientID', 'subtaskID', 'Answer'])

    output = pd.concat([output_1, output_2, output_3], axis=0)
    output.to_csv('output.csv', index=False)

In [ ]:
solve(ctb)

In [ ]:
from sklearn.model_selection import GridSearchCV

# Define the parameter grid for CatBoost
# These are common parameters to tune, adjust ranges as needed
param_grid = {
    'iterations': [100, 200, 300], # Number of boosting rounds
    'learning_rate': [0.01, 0.05, 0.1],
    'depth': [4, 6, 8], # Depth of the trees
    'l2_leaf_reg': [1, 3, 5], # L2 regularization coefficient
    'border_count': [32, 128]
}

# Initialize CatBoostClassifier
# Set random_seed for reproducibility, and verbose to False for cleaner output
cat_model = CatBoostClassifier(random_seed=42, verbose=False, eval_metric='AUC')

# Initialize GridSearchCV
# We use roc_auc as the scoring metric and 3-fold cross-validation
grid_search = GridSearchCV(
    estimator=cat_model,
    param_grid=param_grid,
    cv=3, # Number of cross-validation folds
    scoring='roc_auc',
    n_jobs=-1 # Use all available CPU cores
)

print("Starting GridSearchCV...")
# Fit GridSearchCV to the training data
grid_search.fit(X_train, y_train)
print("GridSearchCV completed.")

# Get the best parameters and best score
best_params = grid_search.best_params_
best_score = grid_search.best_score_

print(f"Best parameters found: {best_params}")
print(f"Best cross-validation ROC AUC score: {best_score}")

# Train the final model with the best parameters
best_ctb_model = CatBoostClassifier(**best_params, random_seed=42, verbose=False, eval_metric='AUC')
best_ctb_model.fit(X_train, y_train)

print("Evaluating the best model on test set...")
evaluate_model(best_ctb_model)

# Update the submission file with predictions from the best model
solve(best_ctb_model)
print("Output CSV updated with predictions from the best model.")